In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from utils import print_trainable_parameters

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

total = torch.cuda.get_device_properties(device).total_memory
free = total - torch.cuda.memory_reserved(device)
print(f"Free: {free/1e9:.2f} GB / Total: {total/1e9:.2f} GB")

from datasets import load_dataset
import os
ds = load_dataset("openai/gsm8k", "main")

total = torch.cuda.get_device_properties(device).total_memory
free = total - torch.cuda.memory_reserved(device)
print(f"Free: {free/1e9:.2f} GB / Total: {total/1e9:.2f} GB")

Using cuda device
Free: 42.41 GB / Total: 42.41 GB
Free: 42.41 GB / Total: 42.41 GB


In [4]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [5]:
train_queries = ds['train'][:10]['question']
train_targets = ds['train'][:10]['answer']

In [6]:
ds = load_dataset("openai/gsm8k", "main")

model_id = "meta-llama/Llama-3.2-1B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

# checkpoint_dir = "no-distil/checkpoint-935"
# lora_model = PeftModel.from_pretrained(base_model, checkpoint_dir)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token   # enable batching

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

: 

In [6]:
dataset_messages = [
    [
        {"role": "system", "content": "Answer math questions step-by-step. End your response by clearly stating your final answer."},
        {"role": "user", "content": q}
    ]
    for q in train_queries
]

pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
pipe.tokenizer.pad_token = pipe.tokenizer.eos_token

print('Starting Inference')
outputs = pipe(dataset_messages, max_new_tokens=350, batch_size=5)

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Inference


Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
dataset_messages = [
    [
        {"role": "system", "content": "Answer math questions step-by-step. End your response by clearly stating your final answer."},
        {"role": "user", "content": q}
    ]
    for q in train_queries
]

model_id =  "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" # "meta-llama/Meta-Llama-3.1-8B-Instruct"
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
pipe.tokenizer.pad_token = pipe.tokenizer.eos_token

print('Starting Inference')
outputs = pipe(dataset_messages, max_new_tokens=350, batch_size=2)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Inference


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializin

: 

In [14]:
[o[0]["generated_text"][-1]['content'] for o in outputs]

['To find out how many clips Natalia sold altogether in April and May, we need to add the number of clips she sold in each month.\n\nIn April, Natalia sold 48 clips.\nIn May, she sold half as many clips, which is 48 / 2 = 24 clips.\n\nSo, the total number of clips Natalia sold in April and May is:\n48 (April) + 24 (May) = 72 clips\n\nTherefore, Natalia sold 72 clips altogether in April and May.',
 "assistant\n\nTo find out how much Weng earned, we need to calculate her hourly wage and multiply it by the number of minutes she worked.\n\nHer hourly wage is $12.\nShe worked for 50 minutes.\n\nFirst, let's convert the time from minutes to hours: 50 minutes / 60 minutes per hour = 0.83 hours.\n\nNow, we multiply her hourly wage by the number of hours she worked: $12/hour * 0.83 hours = $10.04.\n\nSo, Weng earned $10.04.",
 "To find out how much money Betty needs, we need to first find out how much money she has and how much more she needs.\n\nBetty has half of the money she needs, which is 

In [ ]:
extract_last_number = lambda s: float(m.group(1).replace(',', '')) if (m := __import__("re").search(r'(?<!\w)[\$]?(\d+(?:,\d{3})*(?:\.\d+)?)\D*$', s)) else None
[extract_last_number(x[-1]['generated_text'][-1]['content']) for x in outputs]

[72.0, 50.0, 5.0, 54.0, 312.0, 35.0, 48.0, 10.0, 41.0, 180.0]

In [20]:
[extract_last_number(target) for target in train_targets]

[72.0, 10.0, 5.0, 42.0, 624.0, 35.0, 48.0, 16.0, 41.0, 990.0]

In [ ]:

messages = [
    {"role": "system", "content": "Answer the following math question, explicitly stating your step-by-step reasoning. Denote your final answer with \"#### <answer>\""},
    {"role": "user", "content": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"},
]
outputs = pipe(
    messages,
    max_new_tokens=350,
)
print(outputs[0]["generated_text"][-1])

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': 'To find the total number of clips Natalia sold in April and May, we need to add the number of clips sold in each month.\n\nIn April, Natalia sold 48 clips.\nIn May, she sold half as many clips as in April, which is 48 / 2 = 24 clips.\n\nTo find the total number of clips sold in both months, we add the number of clips sold in April and May:\n48 + 24 = 72 clips.\n\n#### 72'}


In [10]:
print(outputs[0]['generated_text'])

[{'role': 'system', 'content': 'Answer the following math question, explicitly stating your step-by-step reasoning. Denote your final answer with "#### <answer>"'}, {'role': 'user', 'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}, {'role': 'assistant', 'content': 'To find the total number of clips Natalia sold in April and May, we need to add the number of clips sold in each month.\n\nIn April, Natalia sold 48 clips.\nIn May, she sold half as many clips as in April, which is 48 / 2 = 24 clips.\n\nTo find the total number of clips sold in both months, we add the number of clips sold in April and May:\n48 + 24 = 72 clips.\n\n#### 72'}]


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("neuralmagic/Llama-2-7b-gsm8k-pruned_70")
model = AutoModelForCausalLM.from_pretrained("neuralmagic/Llama-2-7b-gsm8k-pruned_70", device_map="auto")

input_text = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [13]:
enc = tokenizer(ds['test'][:20]['question'], padding=True, padding_side='left', return_tensors="pt").to(device)
model.eval()
torch.no_grad()
outputs = model.generate(**enc, max_new_tokens=350)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [9]:
single_output = tokenizer.decode(outputs[2], skip_special_tokens=True)
print(single_output)

Marissa is hiking a 12-mile trail. She took 1 hour to walk the first 4 miles, then another hour to walk the next two miles. If she wants her average speed to be 4 miles per hour, what speed (in miles per hour) does she need to walk the remaining distance? First find how many hours Marissa has spent walking so far: 1 hour + 1 hour = <<1+1=2>>2 hours
Then divide the total distance by the number of hours to find Marissa's average speed: 12 miles / 2 hours = <<12/2=6>>6 miles/hour
Then find how far she has to walk to reach her goal speed by subtracting her current speed from her goal speed: 4 miles/hour - 6 miles/hour = <<4-6=2>>2 miles/hour
#### 2


In [10]:
ds['test'][19]['answer']

'First figure out how many hours it takes to hike a 12-mile trail at 4 mph by dividing the distance by the speed: 12 miles / 4 mph = <<12/4=3>>3 hours\nNext subtract the time Marissa already spent walking to find out how much time she has left: 3 hours - 1 hour - 1 hour = <<3-1-1=1>>1 hour\nNow figure out how much distance she has left by subtracting the distance she already traveled from the total distance: 12 miles - 4 miles - 2 miles = <<12-4-2=6>>6 miles\nNow divide the remaining distance by the remaining time to find out how fast in miles per hour Marissa has to travel: 6 miles / 1 hour = <<6/1=6>>6 mph\n#### 6'

In [ ]:
from tqdm.auto import tqdm
for item in range(len(ds['test'])):
    enc = tokenizer(item['question'], return_tensors="pt").to(device)
